# GAT / CSL / M1 - Laplacian eigenvector PE

**Traversal-Based Relative Positional Encodings for Graph Neural Networks**

This notebook trains **one** (architecture, dataset, condition) combination and writes one
JSON file per run to Google Drive. It is fully self-contained - nothing is imported from
another notebook or module.

| | |
|---|---|
| Architecture | `gat` |
| Dataset | `CSL` (150 graphs, 41 nodes, 4-regular and featureless) |
| Condition | `M1` - **baseline**, absolute spectral PE (Dwivedi et al.), with sign-flip augmentation |
| Estimated runtime | ~5 min on a free-Colab T4 |

**Run [`shared/00_setup_and_diagnostic.ipynb`](../../shared/00_setup_and_diagnostic.ipynb)
first** - it verifies the PyG API surface and reports the distance statistics that justify
the radius `r` used here.

Runs are **resumable**: a seed whose JSON file already exists is skipped, so a dropped
Colab session costs one seed rather than the whole notebook.


In [ ]:
# Colab: torch ships preinstalled, torch-geometric does not.
!pip -q install torch-geometric

In [ ]:
# ============================== CONFIGURATION ==============================
ARCH      = "gat"        # "gcn" | "gat"
DATASET   = "CSL"     # torch_geometric.datasets.GNNBenchmarkDataset name
CONDITION = "M1"        # M0 no PE | M1 LapPE | M2 RWPE | M3 T-RPE | M4 T-RPE+#sp
COND_NAME = "Laplacian eigenvector PE"

# --- positional encoding --------------------------------------------------
RADIUS   = 3   # r, the traversal radius used for rewiring (M3/M4 only)
PE_DIM   = 16         # width of the sinusoidal hop-distance embedding
LAP_K    = 8          # number of Laplacian eigenvectors (M1 only)
RW_STEPS = 16         # random-walk length (M2 only)

# --- architecture / optimisation ------------------------------------------
HIDDEN     = 64
LAYERS     = 4
HEADS      = 4        # GAT attention heads (HIDDEN must be divisible by HEADS)
DROPOUT    = 0.0
LR         = 1e-3
EPOCHS     = 100
BATCH_SIZE = 5
PATIENCE   = 30
SEEDS      = [0, 1, 2, 3]

# --- dataset facts (verified by shared/00_setup_and_diagnostic.ipynb) ------
TASK       = "graph"       # "graph" | "node"
N_CLASSES  = 10
NODE_VOCAB = 0   # size of the discrete node-type vocabulary, 0 if features are continuous
N_FOLDS    = 5        # >0 -> stratified k-fold CV (CSL has no predefined split); 0 -> predefined split

print(f"{ARCH.upper()} | {DATASET} | {CONDITION} ({COND_NAME})")

In [ ]:
import os, json, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import GNNBenchmarkDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# Results go to Drive so they survive a session; the datasets go to local disk because
# they are large and freely re-downloadable.
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RESULT_ROOT, DATA_ROOT, IN_COLAB = "/content/drive/MyDrive/trpe", "/content/trpe_data", True
except Exception:
    RESULT_ROOT, DATA_ROOT = os.path.abspath("./trpe"), os.path.abspath("./trpe_data")

RESULTS_DIR = os.path.join(RESULT_ROOT, "results", ARCH, DATASET)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)
print("results ->", RESULTS_DIR)

## The traversal relative positional encoding

For every node $u$ we run a breadth-first traversal to depth $r$, giving the hop distance
$d(u,v)$ to every $v$ within $r$ hops. The rewired edge set is

$$E_r=\{(u,v)\;:\;1\le d(u,v)\le r\},$$

and each rewired edge carries $d(u,v)$ as a *relative* positional feature. The rewiring is
what makes a pairwise encoding meaningful at all: on the original sparse graph every edge
has $d=1$, so a relative encoding would be constant.

**Why matrix products instead of a literal BFS.** A Python BFS per source node means
$O(n)$ interpreter-level traversals per graph, which is far too slow for 45k graphs. The
algebraic form is equivalent and runs in BLAS. Let $A$ be the adjacency matrix; $(A^k)_{uv}$
counts *walks* of length $k$, so the first $k$ with $(A^k)_{uv}>0$ is exactly $d(u,v)$.
And at $k=d(u,v)$ every length-$k$ walk is necessarily a shortest path, so $(A^{d})_{uv}$
is exactly the **number of shortest paths** between $u$ and $v$ - the extra signal that
condition M4 uses. Both quantities therefore fall out of $r$ dense matrix products.


In [ ]:
def compute_trpe(data, r, with_sp_counts):
    """Rewire `data` to its r-hop graph and attach the relative positional encoding.

    Adds:  data.pe_dist [E_r] int64  - hop distance in {1..r}
           data.pe_nsp  [E_r] float  - log(1 + number of shortest paths)  [M4 only]
    """
    n = int(data.num_nodes)
    ei = data.edge_index.numpy()

    A = np.zeros((n, n), dtype=np.float64)
    A[ei[0], ei[1]] = 1.0
    A = np.maximum(A, A.T)              # the benchmark graphs are undirected
    np.fill_diagonal(A, 0.0)

    dist = np.full((n, n), -1, dtype=np.int64)
    np.fill_diagonal(dist, 0)           # d(u,u)=0, excluded from the edge set below
    nsp = np.zeros((n, n), dtype=np.float64)

    Mk = np.eye(n, dtype=np.float64)
    for k in range(1, r + 1):
        Mk = Mk @ A                     # walks of length exactly k
        fresh = (Mk > 0.5) & (dist < 0)  # first time v is reached from u  =>  d(u,v)=k
        if not fresh.any():
            break                       # r already exceeds the diameter
        dist[fresh] = k
        nsp[fresh] = Mk[fresh]          # walks of length d(u,v) are exactly shortest paths

    src, dst = np.nonzero(dist >= 1)    # 1 <= d <= r holds by construction
    data.edge_index = torch.from_numpy(np.stack([src, dst])).long()
    data.pe_dist = torch.from_numpy(dist[src, dst]).long()
    if with_sp_counts:
        data.pe_nsp = torch.from_numpy(np.log1p(nsp[src, dst])).float()
    data.num_nodes = n
    return data


class SinusoidalDistance(nn.Module):
    """Transformer-style sinusoidal embedding of an integer hop distance, so that nearby
    distances get nearby representations instead of arbitrary unrelated ones."""

    def __init__(self, dim):
        super().__init__()
        half = max(1, dim // 2)
        self.register_buffer(
            "freqs", torch.exp(-math.log(10000.0) * torch.arange(half).float() / half)
        )

    def forward(self, d):
        ang = d.float().unsqueeze(-1) * self.freqs
        return torch.cat([ang.sin(), ang.cos()], dim=-1)


class EdgePE(nn.Module):
    """Encodes (hop distance [, shortest-path count]) into a per-edge representation."""

    def __init__(self, pe_dim, use_sp_counts):
        super().__init__()
        self.use_sp_counts = use_sp_counts
        self.sinusoid = SinusoidalDistance(pe_dim)
        in_dim = 2 * max(1, pe_dim // 2) + (1 if use_sp_counts else 0)
        self.mlp = nn.Sequential(nn.Linear(in_dim, pe_dim), nn.ReLU(), nn.Linear(pe_dim, pe_dim))

    def forward(self, pe_dist, pe_nsp=None):
        e = self.sinusoid(pe_dist)
        if self.use_sp_counts:
            e = torch.cat([e, pe_nsp.unsqueeze(-1)], dim=-1)
        return self.mlp(e)

In [ ]:
# ============================ DATASET & CONDITION ============================
USES_EDGE_PE  = CONDITION in ("M3", "M4")   # ours: relative, on rewired edges
USES_SP_COUNT = CONDITION == "M4"
USES_NODE_PE  = CONDITION in ("M1", "M2")   # baselines: absolute, on nodes

if CONDITION == "M1":
    from torch_geometric.transforms import AddLaplacianEigenvectorPE
    NODE_PE_DIM = LAP_K
elif CONDITION == "M2":
    from torch_geometric.transforms import AddRandomWalkPE
    NODE_PE_DIM = RW_STEPS
else:
    NODE_PE_DIM = 0

# LapPE (scipy eigendecomposition) and RWPE are far too slow to recompute on every epoch,
# so they are applied as a pre_transform and cached to disk under a condition-specific
# root. The traversal PE is cheap enough to run on the fly, which also avoids storing
# hundreds of millions of rewired edges.
USE_PRE_TRANSFORM = USES_NODE_PE


def base_features(data):
    """Feature normalisation applied identically under every condition."""
    if DATASET in ("CIFAR10", "MNIST") and getattr(data, "pos", None) is not None:
        # The superpixel coordinate is a genuine input feature, not a positional
        # encoding, so every condition - including the M0 control - receives it.
        data.x = torch.cat([data.x.float(), data.pos.float()], dim=-1)
    if data.x is None:
        data.x = torch.ones((data.num_nodes, 1), dtype=torch.float)
    if data.x.dim() == 1:
        # CLUSTER/PATTERN store a discrete node type rather than a feature vector.
        data.x = F.one_hot(data.x.long().clamp(0, NODE_VOCAB - 1), NODE_VOCAB).float()
    data.x = data.x.float()
    return data


def pe_pre_transform(data):
    data = base_features(data)
    if CONDITION == "M1":
        k = min(LAP_K, max(1, int(data.num_nodes) - 2))
        try:
            data = AddLaplacianEigenvectorPE(k=k, attr_name="node_pe", is_undirected=True)(data)
        except Exception:
            data.node_pe = torch.zeros((data.num_nodes, k))
        if data.node_pe.size(1) < LAP_K:          # pad graphs too small for LAP_K vectors
            pad = LAP_K - data.node_pe.size(1)
            data.node_pe = torch.cat([data.node_pe, torch.zeros((data.num_nodes, pad))], -1)
    elif CONDITION == "M2":
        try:
            data = AddRandomWalkPE(walk_length=RW_STEPS, attr_name="node_pe")(data)
        except Exception:
            data.node_pe = torch.zeros((data.num_nodes, RW_STEPS))
    data.node_pe = data.node_pe.float()
    return data


def runtime_transform(data):
    if USE_PRE_TRANSFORM:
        return data                     # features and node PE were cached at process time
    data = base_features(data)
    if USES_EDGE_PE:
        data = compute_trpe(data, RADIUS, USES_SP_COUNT)
    return data


def _load(split=None):
    root = os.path.join(DATA_ROOT, DATASET + ("_" + CONDITION if USE_PRE_TRANSFORM else ""))
    kw = dict(transform=runtime_transform)
    if USE_PRE_TRANSFORM:
        kw["pre_transform"] = pe_pre_transform
    if split is not None:
        kw["split"] = split
    return GNNBenchmarkDataset(root, DATASET, **kw)


def load_splits():
    """(train, val, test) honouring the predefined split, or (all, None, None) for k-fold."""
    if N_FOLDS > 0:
        return _load(), None, None
    return _load("train"), _load("val"), _load("test")


def fold_indices(labels, n_folds, seed):
    """Stratified k-fold. Fold k is the test set, fold k+1 the validation set."""
    rng = np.random.RandomState(seed)
    labels = np.asarray(labels)
    folds = [[] for _ in range(n_folds)]
    for c in np.unique(labels):
        for i, j in enumerate(rng.permutation(np.where(labels == c)[0])):
            folds[i % n_folds].append(int(j))
    folds = [np.array(sorted(f)) for f in folds]
    for k in range(n_folds):
        v = (k + 1) % n_folds
        yield (np.concatenate([folds[j] for j in range(n_folds) if j not in (k, v)]),
               folds[v], folds[k])

## Model

The relative encoding enters each architecture at its natural point:

* **GAT** - as an additive bias on the attention logit of every rewired edge, via
  `GATConv(edge_dim=...)`. This is what "relative positional encoding" means in the
  transformer sense, so GAT is the encoding's natural home.
* **GCN** - `GCNConv` has no edge-feature channel, so the encoded distance is mapped to a
  learned scalar gate and supplied as `edge_weight`.

Parameter counts are recorded in every result file. The PE modules add roughly 500-1000
parameters to an ~18k-parameter backbone, so no condition holds a meaningful capacity
advantage; the exact counts are reported alongside the accuracies rather than being
forced to match.


In [ ]:
class Net(nn.Module):
    def __init__(self, in_dim, out_dim, hidden, layers):
        super().__init__()
        self.edge_pe = EdgePE(PE_DIM, USES_SP_COUNT) if USES_EDGE_PE else None
        self.gate = nn.Linear(PE_DIM, 1) if (USES_EDGE_PE and ARCH == "gcn") else None

        self.embed = nn.Linear(in_dim + (NODE_PE_DIM if USES_NODE_PE else 0), hidden)
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(layers):
            if ARCH == "gcn":
                self.convs.append(GCNConv(hidden, hidden))
            else:
                self.convs.append(GATConv(hidden, hidden // HEADS, heads=HEADS,
                                          edge_dim=PE_DIM if USES_EDGE_PE else None))
            self.norms.append(nn.BatchNorm1d(hidden))
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, data):
        x = data.x
        if USES_NODE_PE:
            x = torch.cat([x, data.node_pe], dim=-1)
        x = self.embed(x)

        edge_weight = edge_attr = None
        if USES_EDGE_PE:
            pe = self.edge_pe(data.pe_dist, getattr(data, "pe_nsp", None))
            if ARCH == "gcn":
                edge_weight = torch.sigmoid(self.gate(pe)).squeeze(-1)
            else:
                edge_attr = pe

        for conv, norm in zip(self.convs, self.norms):
            h = (conv(x, data.edge_index, edge_weight=edge_weight) if ARCH == "gcn"
                 else conv(x, data.edge_index, edge_attr=edge_attr))
            h = F.relu(norm(h))
            if DROPOUT > 0:
                h = F.dropout(h, p=DROPOUT, training=self.training)
            x = x + h                       # residual, so 4 layers stay trainable

        if TASK == "graph":
            x = global_mean_pool(x, data.batch)
        return self.head(x)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
# ================================ TRAIN / EVALUATE ================================

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def sign_flip(batch):
    """Laplacian eigenvectors are only defined up to sign; randomising the sign each time
    the batch is seen stops the model keying on an arbitrary convention."""
    if CONDITION == "M1" and getattr(batch, "node_pe", None) is not None:
        s = torch.randint(0, 2, (1, batch.node_pe.size(1)), device=batch.node_pe.device)
        batch.node_pe = batch.node_pe * (2.0 * s.float() - 1.0)
    return batch


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    hits = torch.zeros(N_CLASSES); seen = torch.zeros(N_CLASSES)
    for batch in loader:
        batch = batch.to(DEVICE)
        pred = model(batch).argmax(dim=-1)
        y = batch.y.view(-1)
        hit = (pred == y).float().cpu(); y = y.cpu()
        correct += float(hit.sum()); total += y.numel()
        hits += torch.zeros(N_CLASSES).scatter_add_(0, y, hit)
        seen += torch.zeros(N_CLASSES).scatter_add_(0, y, torch.ones_like(hit))
    m = seen > 0
    return correct / max(1, total), float((hits[m] / seen[m]).mean()) if m.any() else 0.0


def run_one(train_ds, val_ds, test_ds, in_dim, seed, tag):
    set_seed(seed)
    nw = 2 if (USES_EDGE_PE and len(train_ds) > 1000) else 0   # hide the on-the-fly RPE cost
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=nw)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=nw)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, num_workers=nw)

    torch.manual_seed(seed)
    model = Net(in_dim, N_CLASSES, HIDDEN, LAYERS).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=10)

    best = {"val_acc": -1.0, "test_acc": 0.0}
    bad, history, t0 = 0, [], time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        tot = 0.0
        for batch in train_loader:
            batch = sign_flip(batch.to(DEVICE))
            opt.zero_grad()
            loss = F.cross_entropy(model(batch), batch.y.view(-1))
            loss.backward(); opt.step()
            tot += float(loss) * batch.num_graphs
        loss_ep = tot / max(1, len(train_ds))

        val_acc, val_bal = evaluate(model, val_loader)
        sched.step(val_acc)
        history.append({"epoch": epoch, "loss": loss_ep, "val_acc": val_acc})

        # Model selection is on VALIDATION accuracy and the test set is read only at the
        # selected epoch. Averaging test accuracy over the final few epochs, as is
        # sometimes done, conflates convergence noise with genuine performance.
        if val_acc > best["val_acc"]:
            test_acc, test_bal = evaluate(model, test_loader)
            best = {"epoch": epoch, "val_acc": val_acc, "val_bal": val_bal,
                    "test_acc": test_acc, "test_bal": test_bal}
            bad = 0
        else:
            bad += 1

        if epoch == 1 or epoch % 5 == 0:
            print(f"  [{tag}] ep {epoch:3d} | loss {loss_ep:.4f} | val {val_acc:.4f} "
                  f"| best val {best['val_acc']:.4f} -> test {best['test_acc']:.4f}")
        if bad >= PATIENCE:
            print(f"  [{tag}] early stop at epoch {epoch}")
            break

    return {**best, "hidden": HIDDEN, "params": count_params(model),
            "epochs_run": epoch, "seconds": round(time.time() - t0, 1), "history": history}

In [ ]:
# ==================================== RUN ====================================
# CSL ships as a single collection of 150 graphs with no predefined split, so the
# benchmark protocol is stratified k-fold cross-validation.
dataset, _, _ = load_splits()
labels = [int(d.y) for d in dataset]
in_dim = dataset[0].x.size(1)
print(f"in_dim={in_dim}  classes={N_CLASSES}  graphs={len(dataset)}")

all_results = []
for seed in SEEDS:
    for fold, (tr_i, va_i, te_i) in enumerate(fold_indices(labels, N_FOLDS, seed)):
        path = os.path.join(RESULTS_DIR, f"{CONDITION}_seed{seed}_fold{fold}.json")
        if os.path.exists(path):
            print(f"seed {seed} fold {fold}: already done, skipping")
            all_results.append(json.load(open(path)));  continue
        print(f"\n=== seed {seed} fold {fold} ===")
        tr = [dataset[int(i)] for i in tr_i]   # 150 tiny graphs: materialise once
        va = [dataset[int(i)] for i in va_i]
        te = [dataset[int(i)] for i in te_i]
        res = run_one(tr, va, te, in_dim, seed, f"s{seed}f{fold}")
        res.update(arch=ARCH, dataset=DATASET, condition=CONDITION, cond_name=COND_NAME,
                   seed=seed, fold=fold, radius=RADIUS if USES_EDGE_PE else None)
        json.dump(res, open(path, "w"), indent=2)
        all_results.append(res)
        print(f"seed {seed} fold {fold}: test acc {res['test_acc']:.4f}")

In [ ]:
# ================================== SUMMARY ==================================
acc = np.array([r["test_acc"] for r in all_results]) * 100
bal = np.array([r["test_bal"] for r in all_results]) * 100
print(f"\n{ARCH.upper()} / {DATASET} / {CONDITION} ({COND_NAME})")
print(f"  runs          : {len(acc)}")
print(f"  test accuracy : {acc.mean():.2f} +/- {acc.std():.2f}")
print(f"  balanced acc  : {bal.mean():.2f} +/- {bal.std():.2f}")
print(f"  parameters    : {all_results[0]['params']:,}")
print(f"  wall clock    : {sum(r['seconds'] for r in all_results) / 60:.1f} min")
print(f"\nJSON in {RESULTS_DIR} - shared/07_results_and_figures.ipynb collects every file.")